In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import kagglehub

In [ ]:
!pip install -q split_folders

In [ ]:
import os
import shutil
import time
import random
import json

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import keras

from tensorflow.keras import layers
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (Dense, Dropout, Flatten, ZeroPadding2D, Conv2D, MaxPooling2D, Activation, GlobalAveragePooling2D, BatchNormalization, DepthwiseConv2D)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import plot_model
from tensorflow.keras.applications import ( DenseNet201, MobileNetV2, EfficientNetB0)
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from sklearn.metrics import ( classification_report, accuracy_score, confusion_matrix)
import splitfolders

Reproducibility

In [ ]:
SEED = 1337

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)

Dataset paths

In [ ]:
DATASET_ROOT = "/kaggle/input/datasets/grassknoted/asl-alphabet"

TRAIN_SOURCE = os.path.join(
    DATASET_ROOT,
    "asl_alphabet_train",
    "asl_alphabet_train"
)

TEST_SOURCE = os.path.join(
    DATASET_ROOT,
    "asl_alphabet_test",
    "asl_alphabet_test"
)

PREPROCESSED_DATASET = "/kaggle/working/asl_preprocessed"

print("Training source:")
print(TRAIN_SOURCE)

print("\nOfficial test source:")
print(TEST_SOURCE)

In [ ]:
class_names = sorted([
    d for d in os.listdir(TRAIN_SOURCE)
    if os.path.isdir(
        os.path.join(TRAIN_SOURCE, d)
    )
])

NUM_CLASSES = len(class_names)

print("Number of classes:", NUM_CLASSES)
print("\nClasses:")
print(class_names)

In [ ]:
total_images = 0

class_counts = {}

for class_name in class_names:

    class_dir = os.path.join(
        TRAIN_SOURCE,
        class_name
    )

    count = len([
        f for f in os.listdir(class_dir)
        if f.lower().endswith(
            (".jpg", ".jpeg", ".png")
        )
    ])

    class_counts[class_name] = count
    total_images += count

print("Total training images:", total_images)

print("\nImages per class:")

for class_name, count in class_counts.items():
    print(f"{class_name:10s}: {count}")

Preprocessing

In [ ]:
def preprocess_image(image_path):

    # Read image
    image = cv2.imread(image_path)

    if image is None:
        return None

    # 1. Grayscale
    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    # 2. Gaussian Blur
    blurred = cv2.GaussianBlur(
        gray,
        (5, 5),
        0
    )

    # 3. Adaptive Threshold
    adaptive = cv2.adaptiveThreshold(
        blurred,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11,
        2
    )

    # 4. Otsu Threshold
    _, otsu = cv2.threshold(
        blurred,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # 5. Combine thresholds
    #
    # The thesis describes bitwise inversion
    # and addition to combine the results.
    adaptive_inv = cv2.bitwise_not(adaptive)

    combined = cv2.add(
        adaptive_inv,
        otsu
    )

    # Convert result into a clean binary image
    _, combined = cv2.threshold(
        combined,
        127,
        255,
        cv2.THRESH_BINARY
    )

    return combined

Test preprocessing on one image

In [ ]:
sample_class = class_names[0]

sample_files = [
    f for f in os.listdir(
        os.path.join(
            TRAIN_SOURCE,
            sample_class
        )
    )
    if f.lower().endswith(
        (".jpg", ".jpeg", ".png")
    )
]

sample_file = sample_files[0]

sample_path = os.path.join(
    TRAIN_SOURCE,
    sample_class,
    sample_file
)

original = cv2.imread(sample_path)

gray = cv2.cvtColor(
    original,
    cv2.COLOR_BGR2GRAY
)

blurred = cv2.GaussianBlur(
    gray,
    (5, 5),
    0
)

adaptive = cv2.adaptiveThreshold(
    blurred,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    11,
    2
)

_, otsu = cv2.threshold(
    blurred,
    0,
    255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

adaptive_inv = cv2.bitwise_not(adaptive)

combined = cv2.add(
    adaptive_inv,
    otsu
)

_, combined = cv2.threshold(
    combined,
    127,
    255,
    cv2.THRESH_BINARY
)

plt.figure(figsize=(20, 5))

plt.subplot(1, 6, 1)
plt.imshow(
    cv2.cvtColor(
        original,
        cv2.COLOR_BGR2RGB
    )
)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 6, 2)
plt.imshow(gray, cmap="gray")
plt.title("Grayscale")
plt.axis("off")

plt.subplot(1, 6, 3)
plt.imshow(blurred, cmap="gray")
plt.title("Gaussian Blur")
plt.axis("off")

plt.subplot(1, 6, 4)
plt.imshow(adaptive, cmap="gray")
plt.title("Adaptive")
plt.axis("off")

plt.subplot(1, 6, 5)
plt.imshow(otsu, cmap="gray")
plt.title("Otsu")
plt.axis("off")

plt.subplot(1, 6, 6)
plt.imshow(combined, cmap="gray")
plt.title("Combined")
plt.axis("off")

plt.show()

Preprocess entire dataset

In [ ]:
if os.path.exists(PREPROCESSED_DATASET):
    shutil.rmtree(PREPROCESSED_DATASET)

os.makedirs(PREPROCESSED_DATASET)

print("Starting preprocessing...\n")

start_time = time.time()

for class_name in class_names:

    source_class_dir = os.path.join(
        TRAIN_SOURCE,
        class_name
    )

    target_class_dir = os.path.join(
        PREPROCESSED_DATASET,
        class_name
    )

    os.makedirs(
        target_class_dir,
        exist_ok=True
    )

    files = [
        f for f in os.listdir(source_class_dir)
        if f.lower().endswith(
            (".jpg", ".jpeg", ".png")
        )
    ]

    print(
        f"Processing {class_name}: "
        f"{len(files)} images"
    )

    for filename in files:

        source_path = os.path.join(
            source_class_dir,
            filename
        )

        target_path = os.path.join(
            target_class_dir,
            filename
        )

        processed = preprocess_image(
            source_path
        )

        if processed is not None:

            cv2.imwrite(
                target_path,
                processed
            )

elapsed = time.time() - start_time

print(
    f"\nPreprocessing completed in "
    f"{elapsed / 60:.2f} minutes."
)

In [ ]:
#Verify preprocessed dataset
processed_total = 0

for class_name in class_names:

    class_dir = os.path.join(
        PREPROCESSED_DATASET,
        class_name
    )

    count = len([
        f for f in os.listdir(class_dir)
        if f.lower().endswith(
            (".jpg", ".jpeg", ".png")
        )
    ])

    processed_total += count

print(
    "Original images:",
    total_images
)

print(
    "Preprocessed images:",
    processed_total
)

Split 70/15/15

In [ ]:
SPLIT_OUTPUT = "/kaggle/working/asl_split"

if os.path.exists(SPLIT_OUTPUT):
    shutil.rmtree(SPLIT_OUTPUT)

splitfolders.ratio(
    PREPROCESSED_DATASET,
    output=SPLIT_OUTPUT,
    seed=SEED,
    ratio=(0.70, 0.15, 0.15),
    group_prefix=None
)

print("Dataset split completed.")

Define split paths

In [ ]:
TRAIN_DIR = os.path.join(
    SPLIT_OUTPUT,
    "train"
)

VAL_DIR = os.path.join(
    SPLIT_OUTPUT,
    "val"
)

TEST_DIR = os.path.join(
    SPLIT_OUTPUT,
    "test"
)

print(TRAIN_DIR)
print(VAL_DIR)
print(TEST_DIR)

Count split images

In [ ]:
def count_dataset_images(directory):

    total = 0

    for class_name in class_names:

        class_dir = os.path.join(
            directory,
            class_name
        )

        if os.path.exists(class_dir):

            total += len([
                f for f in os.listdir(class_dir)
                if f.lower().endswith(
                    (".jpg", ".jpeg", ".png")
                )
            ])

    return total


print(
    "Training images:",
    count_dataset_images(TRAIN_DIR)
)

print(
    "Validation images:",
    count_dataset_images(VAL_DIR)
)

print(
    "Test images:",
    count_dataset_images(TEST_DIR)
)

Common model configuration

In [ ]:
INPUT_SHAPE = (224, 224, 3)
BATCH_SIZE = 24
EPOCHS = 25

LEARNING_RATE = 0.0001

print("Input shape:", INPUT_SHAPE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)

Create pretrained-model generators

In [ ]:
densenet_train_datagen = ImageDataGenerator(
    preprocessing_function=densenet_preprocess
)

densenet_val_datagen = ImageDataGenerator(
    preprocessing_function=densenet_preprocess
)

densenet_test_datagen = ImageDataGenerator(
    preprocessing_function=densenet_preprocess
)

densenet_train_generator = densenet_train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(224, 224),
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=SEED
)

densenet_val_generator = densenet_val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(224, 224),
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

densenet_test_generator = densenet_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(224, 224),
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

DenseNet201 model

In [ ]:
densenet_base = DenseNet201(
    weights="imagenet",
    include_top=False,
    input_shape=INPUT_SHAPE,
    pooling="avg"
)

densenet_base.trainable = True

x = densenet_base.output

x = Dense(
    32,
    activation="relu"
)(x)

densenet_output = Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

densenet_model = Model(
    inputs=densenet_base.input,
    outputs=densenet_output
)

densenet_model.summary()

In [ ]:
earlystop_dense = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

lr_dense = ReduceLROnPlateau(
    monitor="val_loss",
    patience=2,
    verbose=1,
    factor=0.1,
    min_lr=1e-8
)

dense_callbacks = [
    earlystop_dense,
    lr_dense
]

In [ ]:
densenet_model.compile(
    optimizer=Adam(
        learning_rate=LEARNING_RATE
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
dense_start_time = time.time()

dense_history = densenet_model.fit(
    densenet_train_generator,
    validation_data=densenet_val_generator,
    callbacks=dense_callbacks,
    epochs=EPOCHS
)

dense_training_time = (
    time.time() - dense_start_time
)

print(
    f"\nDenseNet201 training time: "
    f"{dense_training_time / 60:.2f} minutes"
)

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    dense_history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    dense_history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.title(
    "DenseNet201 Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    dense_history.history["loss"],
    label="Training Loss"
)

plt.plot(
    dense_history.history["val_loss"],
    label="Validation Loss"
)

plt.title(
    "DenseNet201 Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend()

plt.show()

In [ ]:
densenet_test_generator.reset()

dense_pred_prob = densenet_model.predict(
    densenet_test_generator,
    verbose=1
)

dense_pred = np.argmax(
    dense_pred_prob,
    axis=1
)

dense_true = densenet_test_generator.classes

print(
    "Prediction shape:",
    dense_pred_prob.shape
)

DenseNet201 classification report

In [ ]:
print(
    classification_report(
        dense_true,
        dense_pred,
        target_names=[
            index_to_class[i]
            for i in range(NUM_CLASSES)
        ],
        digits=4
    )
)

In [ ]:
dense_accuracy = accuracy_score(
    dense_true,
    dense_pred
)

print(
    f"DenseNet201 Test Accuracy: "
    f"{dense_accuracy * 100:.4f}%"
)

In [ ]:
dense_cm = confusion_matrix(
    dense_true,
    dense_pred
)

plt.figure(
    figsize=(14, 12)
)

sns.heatmap(
    dense_cm,
    annot=True,
    fmt="d",
    xticklabels=[
        index_to_class[i]
        for i in range(NUM_CLASSES)
    ],
    yticklabels=[
        index_to_class[i]
        for i in range(NUM_CLASSES)
    ]
)

plt.title(
    "DenseNet201 Confusion Matrix"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

DenseNet201 Top-2

In [ ]:
dense_top2 = np.argsort(
    dense_pred_prob,
    axis=1
)[:, -2:]

dense_top2_correct = np.array([
    dense_true[i] in dense_top2[i]
    for i in range(len(dense_true))
])

dense_top2_accuracy = (
    dense_top2_correct.mean() * 100
)

print(
    f"DenseNet201 Top-2 Accuracy: "
    f"{dense_top2_accuracy:.4f}%"
)

DenseNet201 Top-3

In [ ]:
dense_top3 = np.argsort(
    dense_pred_prob,
    axis=1
)[:, -3:]

dense_top3_correct = np.array([
    dense_true[i] in dense_top3[i]
    for i in range(len(dense_true))
])

dense_top3_accuracy = (
    dense_top3_correct.mean() * 100
)

print(
    f"DenseNet201 Top-3 Accuracy: "
    f"{dense_top3_accuracy:.4f}%"
)

DenseNet201 feature maps

In [ ]:
dense_conv_layers = [
    layer
    for layer in densenet_model.layers
    if isinstance(
        layer,
        (
            Conv2D,
            DepthwiseConv2D
        )
    )
]

print(
    "Number of convolutional layers:",
    len(dense_conv_layers)
)

In [ ]:
dense_feature_layer = dense_conv_layers[0]

print(
    "Selected layer:",
    dense_feature_layer.name
)

In [ ]:
dense_feature_input = (
    feature_img * 255.0
)

dense_feature_input = (
    densenet_preprocess(
        dense_feature_input
    )
)

dense_feature_input = np.expand_dims(
    dense_feature_input,
    axis=0
)

dense_feature_model = Model(
    inputs=densenet_model.inputs,
    outputs=dense_feature_layer.output
)

dense_feature_maps = (
    dense_feature_model.predict(
        dense_feature_input
    )
)

print(
    "DenseNet201 feature map shape:",
    dense_feature_maps.shape
)

In [ ]:
num_maps = min(
    dense_feature_maps.shape[-1],
    32
)

plt.figure(
    figsize=(16, 16)
)

for i in range(num_maps):

    ax = plt.subplot(
        4,
        8,
        i + 1
    )

    ax.imshow(
        dense_feature_maps[0, :, :, i],
        cmap="gray"
    )

    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle(
    "DenseNet201 Feature Maps"
)

plt.show()